### CNN (Convolutional Neural Network，卷积神经网络)
CNN是是深度学习领域专门用于处理“网格状数据”（Grid-like Data）的神经网络架构。其广泛应用于图像识别、计算机视觉等领域。其设计理念源于对生物视觉皮层神经机制的模拟，核心原理是通过卷积、池化（下采样）、全连接等操作，自动提取输入数据的层级特征，完成分类或回归任务。

**模型原理**
一个典型卷积神经网络模型通常依次由输入层、卷积层、下采样层、全连接层、输出层等构成。该模型最主要的三个特点是局部感知、权值共享、下采样，局部感知和权值共享体现在卷积层，下采样体现在采样层。输入层即输入数据通常是图像，或其他结构化的数据

<img src="./pictures/cnn_model.jpg" width="1000" align="center" />

**卷积层**
通过卷积核（Filter/Kernel）在输入数据（如图像）上滑动，每次仅与局部区域进行加权求和，模拟生物视觉的局部感知机制。如果输入数据为一张灰度图，那么卷积核为二维张量（如3×3），对应计算输入图像的值；如果输入数据为一张 RGB 图像（3 通道），则卷积核通常为三维张量（如 3×3×3），对应计算输入图像的多通道值。

同一个卷积核在滑动过程中参数（权重）保持不变，例如，一个输入图像用 3×3 的卷积核提取特征，只需学习 9 个参数，大幅减少模型参数量。卷积运算的结果称为特征映射，每个特征映射对应输入数据的一种特征（如边缘、纹理、颜色等）。该层操作有卷积核大小、步长（Stride）、填充（Padding）等关键参数。卷积核大小常见有 3×3、5×5 等；步长（Stride）是卷积核滑动的间隔，步长为 1 时逐像素滑动，步长为 2 时每隔一个像素滑动；填充（Padding）是指在输入数据边缘补零，保持特征图尺寸（如 “Same Padding” 使输出尺寸与输入相同）或控制尺寸缩小比例。通过叠加多个卷积核，可提取多维度特征。该层的输出通常叫做特征图（fature map）。

**下采样层**
下采样（Downsampling）也叫做池化（Pooling）。该层的一个作用是实现下采样，通过聚合局部区域特征（如取最大值或平均值），缩小特征图尺寸，减少计算量。另一个作用是实现平移不变性，使特征对输入数据的微小位移不敏感，增强模型鲁棒性。同样，该层的输出也可叫做特征图。

**全连接层**
经过多层卷积和池化操作后，特征图被展平为一维向量，输入到全连接层（Fully Connected Layer）。全连接层中的每个神经元与上一层的所有神经元相连，通过矩阵乘法和激活函数，将提取到的特征映射到预测输出。

**输出层**
经过多层计算之后，输出预测内容。对于分类任务，通常使用 Softmax 激活函数，输出各类别的概率分布；对于回归任务，通常可直接设计为一个以线性函数为激活函数的神经元，输出连续型变量值。


**网络深度**
- 低层级卷积层提取边缘、颜色、纹理等基础特征
- 高层级通过多层卷积和池化，组合基础特征形成复杂语义（如 “眼睛”→“人脸”→“物体类别”）
- 网络深度增加（层数增多），可提取更抽象的特征，但需解决梯度消失 / 爆炸问题（通过残差连接、批量归一化等技术优化）

**应用领域**
- **图像领域**：分类、目标检测（如 YOLO、Faster R-CNN）、语义分割（如 U-Net）
- **视频领域**：行为识别（3D CNN）、视频生成
- **其他领域**：自然语言处理（文本 CNN）、医学影像分析、自动驾驶等

In [6]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import matplotlib
matplotlib.use('TkAgg')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# -------------------- 1. 生成数据 --------------------
def generate_stripe_data(num_samples=4000, img_size=16):
    """
    生成条纹图像数据集。
    返回：图像 (batch, 1, img_size, img_size) 和标签 (batch,)
    标签：0 = 横向条纹，1 = 纵向条纹
    """
    images = []
    labels = []
    for _ in range(num_samples):
        # 随机选择条纹方向
        label = np.random.randint(0, 2)  # 0或1
        # 创建全零图像
        img = np.zeros((img_size, img_size), dtype=np.float32)
        if label == 0:   # 横向条纹：每隔一行填255
            for i in range(0, img_size, 2):
                img[i, :] = 255.0
        else:            # 纵向条纹：每隔一列填255
            for j in range(0, img_size, 2):
                img[:, j] = 255.0
        # 归一化到 [0,1]
        img = img / 255.0
        images.append(img)
        labels.append(label)
    
    # 转为张量
    X = torch.FloatTensor(images).unsqueeze(1)  # (样本, 1, 高, 宽)
    y = torch.LongTensor(labels)
    
    # 划分训练集和测试集
    split = int(0.8 * num_samples)
    train_data = TensorDataset(X[:split], y[:split])
    test_data = TensorDataset(X[split:], y[split:])
    return train_data, test_data

# -------------------- 2. 定义一个简单的CNN --------------------
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 第一个卷积层：输入通道1，输出通道8，卷积核3x3，填充1保持尺寸
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        # 池化层：2x2最大池化，尺寸减半
        self.pool = nn.MaxPool2d(2, 2)
        # 第二个卷积层：输入8，输出16，卷积核3x3
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        # 全连接层：经过两次池化，图像尺寸从16->8->4，所以特征图为 16 * 4 * 4 = 256
        self.fc1 = nn.Linear(16 * 4 * 4, 64)
        self.fc2 = nn.Linear(64, 2)   # 二分类输出

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))   # 16->8
        x = self.pool(torch.relu(self.conv2(x)))   # 8->4
        x = x.view(x.size(0), -1)                  # 展平
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)                            # 不经过softmax，因为CrossEntropyLoss自带
        return x

# -------------------- 3. 训练函数 --------------------
def train_model(model, train_loader, test_loader, criterion, optimizer, epochs=30):
    train_losses, test_accs = [], []
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        train_losses.append(avg_loss)

        # 测试
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                pred = model(X_batch)
                _, predicted = torch.max(pred, 1)
                total += y_batch.size(0)
                correct += (predicted == y_batch).sum().item()
        acc = correct / total
        test_accs.append(acc)
        print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Test Acc: {acc*100:.2f}%')
    return train_losses, test_accs

# -------------------- 4. 可视化 --------------------
def plot_results(losses, accs):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(losses)
    ax1.set_title('Training Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')

    ax2.plot(accs)
    ax2.set_title('Test Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

def show_sample_images(train_data, num=8):
    """显示几张训练样本，帮助理解数据"""
    fig, axes = plt.subplots(2, 4, figsize=(8, 4))
    for i in range(num):
        img, label = train_data[i]
        ax = axes[i//4, i%4]
        ax.imshow(img.squeeze(), cmap='gray')
        ax.set_title('Horizontal' if label==0 else 'Vertical')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# -------------------- 5. 主程序 --------------------
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # 超参数
    batch_size = 64
    learning_rate = 0.001
    epochs = 30

    # 生成数据
    train_data, test_data = generate_stripe_data()
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=batch_size)

    # 显示一些样本（直观了解数据）
    show_sample_images(train_data)

    # 创建模型
    model = SimpleCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # 训练
    print("开始训练...")
    losses, accs = train_model(model, train_loader, test_loader, criterion, optimizer, epochs)

    # 绘制训练曲线
    plot_results(losses, accs)

    # 手动测试：预测一张新生成的条纹图
    model.eval()
    # 生成一个横向条纹
    h_img = np.zeros((16, 16), dtype=np.float32)
    for i in range(0, 16, 2):
        h_img[i, :] = 1.0
    # 生成一个纵向条纹
    v_img = np.zeros((16, 16), dtype=np.float32)
    for j in range(0, 16, 2):
        v_img[:, j] = 1.0

    for name, img in [('Horizontal', h_img), ('Vertical', v_img)]:
        x = torch.FloatTensor(img).unsqueeze(0).unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(x)
            pred = torch.argmax(output, dim=1).item()
        print(f"{name} image -> predicted as: {'Horizontal' if pred==0 else 'Vertical'}")

ModuleNotFoundError: No module named 'torch'